In [2]:
import tensorflow as tf
tf.keras.backend.clear_session()

import os
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight

In [3]:
BASE_DIR = r"C:/Users/raksh/x-ai for medical imaging/data/knee_xray"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

MODEL_SAVE_PATH = "backend/saved_models/knee_model.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16   # 🔥 smaller batch → better generalization
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 10

In [4]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=12,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.15,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_gen.num_classes
print("Classes:", train_gen.class_indices)

Found 1070 images belonging to 2 classes.
Found 240 images belonging to 2 classes.
Found 240 images belonging to 2 classes.
Classes: {'NORMAL': 0, 'OSTEOPOROSIS': 1}


In [5]:
labels = train_gen.classes

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

Class Weights: {0: np.float64(1.0), 1: np.float64(1.0)}


In [6]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.5)(x)   # 🔥 strong dropout

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,218,917 (16.09 MB)

 Trainable params: 166,786 (651.51 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),  # slightly lower
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "backend/saved_models/knee_phase1.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",   # 🔥 better for small data
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

In [10]:
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/15
67/67 ━━━━━━━━━━━━━━━━━━━━ 68s 789ms/step - accuracy: 0.7019 - loss: 0.7750 - val_accuracy: 0.7000 - val_loss: 0.5967 - learning_rate: 5.0000e-04
Epoch 2/15
67/67 ━━━━━━━━━━━━━━━━━━━━ 49s 723ms/step - accuracy: 0.7430 - loss: 0.6319 - val_accuracy: 0.7250 - val_loss: 0.5285 - learning_rate: 5.0000e-04
Epoch 3/15
67/67 ━━━━━━━━━━━━━━━━━━━━ 48s 712ms/step - accuracy: 0.7673 - loss: 0.5399 - val_accuracy: 0.6583 - val_loss: 0.6594 - learning_rate: 5.0000e-04
Epoch 4/15
67/67 ━━━━━━━━━━━━━━━━━━━━ 49s 724ms/step - accuracy: 0.8047 - loss: 0.4447 - val_accuracy: 0.7458 - val_loss: 0.6530 - learning_rate: 5.0000e-04
Epoch 5/15
67/67 ━━━━━━━━━━━━━━━━━━━━ 51s 766ms/step - accuracy: 0.7963 - loss: 0.4344 - val_accuracy: 0.7417 - val_loss: 0.7070 - learning_rate: 1.5000e-04
Epoch 6/15
67/67 ━━━━━━━━━━━━━━━━━━━━ 50s 745ms/step - accuracy: 0.8037 - loss: 0.4425 - val_accuracy: 0.7708 - val_loss: 0.7184 - learning_rate: 1.5000e-04


In [11]:
base_model.trainable = True

for layer in base_model.layers[:-80]:   # 🔥 freeze more layers
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [12]:
callbacks_ft = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

In [13]:
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE2,
    class_weight=class_weights,
    callbacks=callbacks_ft
)

Epoch 1/10
67/67 ━━━━━━━━━━━━━━━━━━━━ 133s 1s/step - accuracy: 0.6215 - loss: 0.8171 - val_accuracy: 0.7833 - val_loss: 0.4977 - learning_rate: 1.0000e-05
Epoch 2/10
67/67 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0.6888 - loss: 0.6772 - val_accuracy: 0.7833 - val_loss: 0.5290 - learning_rate: 1.0000e-05
Epoch 3/10
67/67 ━━━━━━━━━━━━━━━━━━━━ 61s 907ms/step - accuracy: 0.7028 - loss: 0.6637 - val_accuracy: 0.7792 - val_loss: 0.5724 - learning_rate: 1.0000e-05
Epoch 4/10
67/67 ━━━━━━━━━━━━━━━━━━━━ 61s 908ms/step - accuracy: 0.6850 - loss: 0.7147 - val_accuracy: 0.7625 - val_loss: 0.6066 - learning_rate: 3.0000e-06
Epoch 5/10
67/67 ━━━━━━━━━━━━━━━━━━━━ 62s 924ms/step - accuracy: 0.7056 - loss: 0.6517 - val_accuracy: 0.7500 - val_loss: 0.6239 - learning_rate: 3.0000e-06


In [14]:
test_loss, test_acc = model.evaluate(test_gen)
print("✅ Knee Model Test Accuracy:", test_acc)

15/15 ━━━━━━━━━━━━━━━━━━━━ 11s 650ms/step - accuracy: 0.7542 - loss: 0.5032
✅ Knee Model Test Accuracy: 0.7541666626930237


In [15]:
from tensorflow.keras.models import load_model

# Load best Phase 1 model
model = load_model("backend/saved_models/knee_phase1.keras")

In [16]:
test_loss, test_acc = model.evaluate(test_gen)
print("✅ Phase 1 Test Accuracy:", test_acc)

15/15 ━━━━━━━━━━━━━━━━━━━━ 18s 578ms/step - accuracy: 0.8333 - loss: 0.3972
✅ Phase 1 Test Accuracy: 0.8333333134651184


In [17]:
model.save("backend/saved_models/knee_final.keras")

print("🔥 Final model saved as knee_final.keras")

🔥 Final model saved as knee_final.keras
